# Chapter 2 — NB 04: anotação funcional por COORDENADA (VEP + dbNSFP4.5a), **sem Biofilter**

**Objetivo:** anotar as variantes da união por **coordenada** (não por DB de variantes conhecidas), cobrindo **todas** — inclusive as privadas que o BF4 marcou `not_found`. Depois, num passo separado, comparamos onde o VEP superou o BF4.

**Pipeline (LSF):** `scripts/04/run_vep_annovar_annotation.sh` + `submit_vep_annovar.sh`
- **VEP** (`ensemblvep/109.1`, cache 109, `--pick`) → `Consequence` + `IMPACT` (HIGH = pLOF). Anota por coordenada → cobertura total.
- **ANNOVAR + dbNSFP4.5a (canonical)** → `AlphaMissense_score`/`_pred` (pré-computado para toda missense possível).

> Por que isto bate o BF4: o BF4 só anota variantes **conhecidas (gnomAD)** → perdeu 75% (privadas) e não tinha AlphaMissense para o ZNF175. VEP/dbNSFP anotam **qualquer** variante pela posição.

In [1]:
# --- Setup ---
from pathlib import Path
from io import StringIO
import pandas as pd
import numpy as np

BASE      = Path("/project/hall/analysis/hearing-loss-genomics")
R04       = BASE / "analysis/chapter_2/results/04"
R02       = BASE / "analysis/chapter_2/results/02"
R03       = BASE / "analysis/chapter_2/results/03"

union = pd.read_csv(R02 / "cmp_union_all_variants.csv")
print("union variants:", len(union))

union variants: 386


## 1. VEP — consequência + IMPACT (pLOF) para TODAS as variantes
O `.tab` do VEP tem linhas `##` de cabeçalho; a linha de colunas começa com `#Uploaded_variation`.

In [2]:
# --- Load VEP ---
lines = [l for l in open(R04 / "znf175_vep.tab") if not l.startswith("##")]
vep = pd.read_csv(StringIO("".join(lines)), sep="\t")
vep.columns = [c.lstrip("#") for c in vep.columns]
print("VEP rows:", len(vep))
print(vep["Consequence"].value_counts().head(12).to_string())
print("\nIMPACT:", dict(vep["IMPACT"].value_counts()))

VEP rows: 384
Consequence
missense_variant                                                            169
intron_variant                                                               90
synonymous_variant                                                           58
5_prime_UTR_variant                                                          18
3_prime_UTR_variant                                                          17
frameshift_variant                                                           14
stop_gained                                                                   9
splice_polypyrimidine_tract_variant,intron_variant                            2
splice_region_variant,synonymous_variant                                      2
splice_donor_region_variant,intron_variant                                    2
splice_region_variant,intron_variant                                          1
splice_region_variant,splice_polypyrimidine_tract_variant,intron_variant      1

IMPACT: {'MOD

## 2. ANNOVAR / dbNSFP4.5a — AlphaMissense (missense)
Casamos por `(POS,REF,ALT)` apenas para SNVs (AlphaMissense é missense → SNV).

In [3]:
# --- Load ANNOVAR AlphaMissense ---
ann = pd.read_csv(R04 / "znf175_annovar.hg38_multianno.txt", sep="\t", dtype=str, na_values=".")
am_cols = [c for c in ann.columns if "AlphaMissense" in c]
print("AlphaMissense cols:", am_cols)
snv = ann[(ann["Ref"].str.len()==1) & (ann["Alt"].str.len()==1)].copy()
snv["amkey"] = snv["Start"].astype(str) + ":" + snv["Ref"] + ":" + snv["Alt"]
snv["AlphaMissense_score"] = pd.to_numeric(snv["AlphaMissense_score"], errors="coerce")
print("SNVs with AlphaMissense_score:", snv["AlphaMissense_score"].notna().sum())

AlphaMissense cols: ['AlphaMissense_score', 'AlphaMissense_rankscore', 'AlphaMissense_pred']
SNVs with AlphaMissense_score: 177


## 3. Anexar VEP + AlphaMissense à tabela união

In [4]:
# --- Merge VEP + AM onto union ---
u = union.copy()
u["vep_key"] = "19:" + u["key"]
u = u.merge(vep[["Uploaded_variation","Consequence","IMPACT","SYMBOL","BIOTYPE","CANONICAL"]],
            left_on="vep_key", right_on="Uploaded_variation", how="left")
u = u.merge(snv[["amkey","AlphaMissense_score","AlphaMissense_pred"]],
            left_on="key", right_on="amkey", how="left")

# flags
u["is_pLOF_vep"]   = (u["IMPACT"] == "HIGH")
u["is_missense"]   = u["Consequence"].astype(str).str.contains("missense")

keep = ["key","CHROM","POS","REF","ALT","group","present_v1","present_v2","MAF_v1","MAF_v2",
        "Consequence","IMPACT","is_pLOF_vep","is_missense",
        "AlphaMissense_score","AlphaMissense_pred","SYMBOL","CANONICAL"]
u[keep].to_csv(R04 / "cmp_union_annotated_vep.csv", index=False)
print("saved -> results/04/cmp_union_annotated_vep.csv")
print("\nVEP consequence coverage:", u["Consequence"].notna().sum(), "/", len(u))
print("pLOF (IMPACT=HIGH):", int(u["is_pLOF_vep"].sum()))
print(u[u["is_pLOF_vep"]][["key","Consequence","MAF_v1","MAF_v2"]].to_string(index=False))

saved -> results/04/cmp_union_annotated_vep.csv

VEP consequence coverage: 384 / 386
pLOF (IMPACT=HIGH): 24
                key        Consequence   MAF_v1   MAF_v2
      51573356:GA:G frameshift_variant      NaN 0.000011
      51581437:A:AG frameshift_variant 0.000611 0.001098
       51581490:G:T        stop_gained 0.000044 0.000011
      51581812:C:CA frameshift_variant      NaN 0.000011
      51586647:A:AT frameshift_variant      NaN 0.000011
     51586846:ATT:A frameshift_variant      NaN 0.000011
       51587087:C:A        stop_gained      NaN 0.000011
      51587209:AC:A frameshift_variant      NaN 0.000011
       51587346:G:T        stop_gained      NaN 0.000023
     51587396:AAC:A frameshift_variant      NaN 0.000011
      51587522:CA:C frameshift_variant      NaN 0.000011
       51587583:C:T        stop_gained      NaN 0.000057
     51587656:CAG:C frameshift_variant      NaN 0.000034
   51587727:CAAAG:C frameshift_variant 0.000131 0.000080
       51587771:T:A        stop_gaine

## 4. Cobertura VEP vs BF4 (prévia — comparação completa fica para depois)

In [5]:
# --- Coverage vs BF4 (preview) ---
bf4 = pd.read_csv(R03 / "cmp_union_annotated_bf4.csv")
n   = len(u)
vep_cov = int(u["Consequence"].notna().sum())
am_n    = int(u["AlphaMissense_score"].notna().sum())
bf4_found = int((bf4["bf4_status"]=="found").sum()) if "bf4_status" in bf4 else int((bf4.get("status")=="found").sum())

print(f"{'metric':30s} {'VEP/dbNSFP':>12s} {'BF4':>8s}")
print(f"{'variants annotated (conseq.)':30s} {vep_cov:>12d} {bf4_found:>8d}")
print(f"{'AlphaMissense scores':30s} {am_n:>12d} {0:>8d}")
print(f"{'pLOF flagged':30s} {int(u['is_pLOF_vep'].sum()):>12d} {6:>8d}")

# found-rate por raridade (VEP deve ser ~100% em todas)
u["maf_max"] = u[["MAF_v1","MAF_v2"]].max(axis=1)
sub = u[u["ALT"]!="*"].copy()
sub["bin"] = pd.cut(sub["maf_max"], [0,1e-4,1e-3,1e-2,1], labels=["<0.01%","0.01-0.1%","0.1-1%",">=1%"])
sub["vep_ann"] = sub["Consequence"].notna()
print("\nVEP annotation-rate por faixa de raridade (vs BF4 11% nas <0.01%):")
print(sub.groupby("bin", observed=True)["vep_ann"].mean().round(2).to_string())

metric                           VEP/dbNSFP      BF4
variants annotated (conseq.)            384       97
AlphaMissense scores                    177        0
pLOF flagged                             24        6

VEP annotation-rate por faixa de raridade (vs BF4 11% nas <0.01%):
bin
<0.01%       1.0
0.01-0.1%    1.0
0.1-1%       1.0
>=1%         1.0


In [6]:
# --- Summary ---
summary = f'''# NB 04 — VEP + dbNSFP4.5a annotation of the ZNF175 union (no Biofilter)

Pipeline: VEP (cache 109, --pick) for consequence+IMPACT; ANNOVAR dbNSFP4.5a(canonical) for AlphaMissense.
Annotates by COORDINATE -> covers ALL variants (incl. private). Output: cmp_union_annotated_vep.csv.

## Coverage
- VEP consequence: {vep_cov}/{n} variants annotated (vs BF4 found {bf4_found}/{n})
- pLOF (IMPACT=HIGH): {int(u["is_pLOF_vep"].sum())} (vs BF4 6)
- AlphaMissense scores: {am_n} (vs BF4 0 for ZNF175)

## Why VEP beats BF4 here
- BF4 = gnomAD-known-variant DB -> 75% not_found (private/ultra-rare escape; ~11% found <0.01% MAF).
- VEP/dbNSFP compute consequence/AlphaMissense from the GENOME by coordinate -> ~100% coverage at every rarity.

## Next
- NB 05 (planned): full VEP-vs-BF4 reconciliation -- agreement on shared variants, what each adds,
  and whether LOFTEE HC/LC (needs human_ancestor/gerp data) changes the pLOF set.

## Outputs
- cmp_union_annotated_vep.csv, znf175_vep.tab, znf175_annovar.hg38_multianno.txt, znf175_union.vcf
'''
(R04 / "nb04_vep_annotation_summary.md").write_text(summary)
print(summary)

# NB 04 — VEP + dbNSFP4.5a annotation of the ZNF175 union (no Biofilter)

Pipeline: VEP (cache 109, --pick) for consequence+IMPACT; ANNOVAR dbNSFP4.5a(canonical) for AlphaMissense.
Annotates by COORDINATE -> covers ALL variants (incl. private). Output: cmp_union_annotated_vep.csv.

## Coverage
- VEP consequence: 384/386 variants annotated (vs BF4 found 97/386)
- pLOF (IMPACT=HIGH): 24 (vs BF4 6)
- AlphaMissense scores: 177 (vs BF4 0 for ZNF175)

## Why VEP beats BF4 here
- BF4 = gnomAD-known-variant DB -> 75% not_found (private/ultra-rare escape; ~11% found <0.01% MAF).
- VEP/dbNSFP compute consequence/AlphaMissense from the GENOME by coordinate -> ~100% coverage at every rarity.

## Next
- NB 05 (planned): full VEP-vs-BF4 reconciliation -- agreement on shared variants, what each adds,
  and whether LOFTEE HC/LC (needs human_ancestor/gerp data) changes the pLOF set.

## Outputs
- cmp_union_annotated_vep.csv, znf175_vep.tab, znf175_annovar.hg38_multianno.txt, znf175_union.vcf

